# MILE-Inspired World Model Policy

## Why world models

Both BC MLP and BEV CNN are **reactive** policies: they map the current observation
directly to a trajectory without any explicit model of how the world evolves.
This is efficient but has a fundamental limitation: they cannot *imagine* consequences.

A world model adds a second component:

    Encoder   :  observation_t  →  latent_t  (z_t)
    World model:  (z_t, action_t) →  z_{t+1}   (transition in latent space)
    Policy    :  z_t            →  trajectory_t

The world model is trained with a **consistency loss** that forces the latent
transition to match the actual encoded next state. This means z encodes not just
"what is the current state" but "how will the state change given this action."

## What this notebook builds

A minimal MILE-style architecture trained on nuPlan ego state sequences:

- **Encoder**: 6-dim ego state → 64-dim latent z
- **World model**: GRU(z_t, a_t) → z_{t+1}
  where a_t = (dx, dy, d_yaw) from the trajectory (scalar, not image)
- **Policy**: z_t → 48-dim trajectory (16 steps × 3 dims)
- **Joint training**: L_total = L_imitation + β × L_consistency

The consistency loss is the key: it trains the world model to correctly predict
the next latent given the current latent and action. At inference, the policy
reads z_t and outputs the trajectory directly (no explicit world model rollout).
Model-based planning (rolling out the world model for MCTS / CEM) is noted
as a future extension.

## Comparison to MILE paper (Hu et al. 2022, arXiv:2209.14430)

| Component | This notebook | Full MILE |
|---|---|---|
| Input | 6-dim ego state | BEV image + LiDAR |
| Encoder | MLP | ConvRNN |
| World model | GRU cell | Temporal transformer |
| World model loss | L2 consistency | ELBO (VAE) |
| Planning | Feedforward policy | CEM imagined rollout |

This is "MILE-inspired" — the world model concept and joint training objective
are the same. We simplify the input representation to stay within nuPlan's
available ego-state features without a full BEV rasterizer.

**REF:** Hu et al. (2022) "Model-Based Imitation Learning for Urban Driving."
arXiv:2209.14430. NeurIPS 2022.


In [ ]:
# Cell 1 — Imports and config
import os, sys, sqlite3
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

sys.path.insert(0, '/Users/parvpatodia/nuplan-devkit')
sys.path.insert(0, '/Users/parvpatodia/Desktop/diffusion-policy-zoo/nuplan')

os.environ.setdefault('NUPLAN_DATA_ROOT', '/Users/parvpatodia/nuplan-devkit/data/cache')
os.environ.setdefault('NUPLAN_MAPS_ROOT', '/Users/parvpatodia/nuplan-devkit/maps')
os.environ.setdefault('NUPLAN_EXP_ROOT',  '/Users/parvpatodia/nuplan-devkit/exp')
os.environ.setdefault('NUPLAN_TUTORIAL_PATH', '/Users/parvpatodia/nuplan-devkit/tutorials')

FUTURE_STEPS = 16
DT           = 0.1   # nuPlan 10 Hz
LATENT_DIM   = 64    # world model latent dimension
BETA         = 0.5   # WHY: consistency loss weight. MILE paper uses 1.0 but
                     # with scalar state (vs BEV), consistency converges faster.
                     # 0.5 prevents the consistency loss from dominating imitation.
STRIDE       = 10    # same as BC pipeline (same data density)

DB_DIR    = Path('/Users/parvpatodia/nuplan-devkit/data/cache/mini')
CKPT_BC   = Path('checkpoints/bc_best.pt')        # BC MLP baseline
CKPT_MILE = Path('checkpoints/mile_policy.pt')    # this notebook writes this
DEVICE    = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

print(f'Device:     {DEVICE}')
print(f'Latent dim: {LATENT_DIM}')
print(f'Beta:       {BETA}  (consistency loss weight)')
print(f'DB files:   {len(list(DB_DIR.glob("*.db")))}')


In [ ]:
# Cell 2 — MILEDataset
#
# Each training sample contains:
#   state_t        : (6,)    ego features at time t  [sin(y), cos(y), vx, vy, ax, ay]
#   future_states  : (16, 6) ego features at t+1 ... t+16
#   future_traj    : (16, 3) expert trajectory deltas (dx, dy, d_yaw), ego-frame
#
# WHY future_states (not just future_traj):
#   The consistency loss requires encoding the ACTUAL next state (z_{t+j+1} = encoder(state_{t+j+1}))
#   and comparing it to the PREDICTED next state (z_pred = world_model(z_{t+j}, a_j)).
#   Without future_states, there is nothing to compare the world model prediction against.
#
# Memory: (N, 16, 6) future_states + (N, 6) state_t + (N, 48) traj.
#   With N=260K: 260K × (96 + 6 + 48) × 4 bytes ≈ 156 MB. Fits in RAM.
#   WHY pre-extract (vs on-the-fly): consistency loss needs exact future ego states
#   which don't require any rasterization — pure numpy, fast extraction.

def quat_to_yaw(qw, qx, qy, qz):
    return np.arctan2(2.0 * (qw * qz + qx * qy), 1.0 - 2.0 * (qy**2 + qz**2))

def ego_features(arr: np.ndarray, i: int) -> np.ndarray:
    """Extract 6-dim feature vector at row i: [sin(yaw), cos(yaw), vx, vy, ax, ay]"""
    return np.array([
        np.sin(arr[i, 2]), np.cos(arr[i, 2]),
        arr[i, 3], arr[i, 4],   # vx, vy
        arr[i, 5], arr[i, 6],   # ax, ay
    ], dtype=np.float32)

def relative_pose(arr: np.ndarray, i: int, anchor_x: float, anchor_y: float, anchor_yaw: float):
    """(dx, dy, d_yaw) of row i relative to anchor pose, ego-frame."""
    cos_h = np.cos(-anchor_yaw)
    sin_h = np.sin(-anchor_yaw)
    dx_w  = arr[i, 0] - anchor_x
    dy_w  = arr[i, 1] - anchor_y
    dx_e  =  cos_h * dx_w - sin_h * dy_w
    dy_e  =  sin_h * dx_w + cos_h * dy_w
    d_yaw = arr[i, 2] - anchor_yaw
    d_yaw = (d_yaw + np.pi) % (2 * np.pi) - np.pi
    return dx_e, dy_e, d_yaw

def extract_mile_windows(db_path: str, stride: int = STRIDE):
    conn = sqlite3.connect(db_path)
    rows = conn.execute(
        'SELECT x, y, qw, qx, qy, qz, vx, vy, acceleration_x, acceleration_y '
        'FROM ego_pose ORDER BY timestamp'
    ).fetchall()
    conn.close()
    if len(rows) < FUTURE_STEPS + 1:
        return None, None, None

    arr = np.array(rows, dtype=np.float64)
    arr_yaw = quat_to_yaw(arr[:,2], arr[:,3], arr[:,4], arr[:,5])
    # Rebuild arr with yaw in place of quaternion: [x, y, yaw, vx, vy, ax, ay]
    arr = np.column_stack([arr[:,0], arr[:,1], arr_yaw,
                           arr[:,6], arr[:,7], arr[:,8], arr[:,9]]).astype(np.float32)
    N = len(arr)

    states_list, futures_list, traj_list = [], [], []
    for i in range(0, N - FUTURE_STEPS, stride):
        # Current state
        s_t = ego_features(arr, i)

        # Future ego states (for consistency loss)
        fut_states = np.array([ego_features(arr, i + j + 1) for j in range(FUTURE_STEPS)],
                              dtype=np.float32)

        # Future trajectory (for imitation loss)
        cx, cy, cyaw = arr[i, 0], arr[i, 1], arr[i, 2]
        traj = np.zeros(FUTURE_STEPS * 3, dtype=np.float32)
        for j in range(FUTURE_STEPS):
            dx_e, dy_e, d_yaw = relative_pose(arr, i + j + 1, cx, cy, cyaw)
            traj[j * 3]     = dx_e
            traj[j * 3 + 1] = dy_e
            traj[j * 3 + 2] = d_yaw

        states_list.append(s_t)
        futures_list.append(fut_states)
        traj_list.append(traj)

    return (np.array(states_list,  dtype=np.float32),   # (N, 6)
            np.array(futures_list, dtype=np.float32),   # (N, 16, 6)
            np.array(traj_list,    dtype=np.float32))   # (N, 48)


# ── Extract from all 64 DB files ──────────────────────────────────────────────
from tqdm import tqdm

all_S, all_F, all_T = [], [], []
db_files = sorted(DB_DIR.glob('*.db'))
for db_path in tqdm(db_files, desc='Extracting', ncols=80):
    S, F, T = extract_mile_windows(str(db_path))
    if S is not None:
        all_S.append(S); all_F.append(F); all_T.append(T)

X_states  = np.concatenate(all_S, axis=0)   # (N, 6)
X_futures = np.concatenate(all_F, axis=0)   # (N, 16, 6)
Y_traj    = np.concatenate(all_T, axis=0)   # (N, 48)

print(f'Total windows:     {X_states.shape[0]:,}')
print(f'State shape:       {X_states.shape}')
print(f'Future states:     {X_futures.shape}')
print(f'Trajectory shape:  {Y_traj.shape}')


In [ ]:
# Cell 3 — Normalization
#
# WHY normalize all 6-dim state features (including future states):
#   The consistency loss compares z_pred = world_model(z_t, a_t) with
#   z_actual = encoder(state_{t+1}). Both pass through the SAME encoder,
#   so the normalisation statistics must be identical for state_t and
#   state_{t+j+1}. We use the global mean/std across all extracted states.
#
# The trajectory normalisation is per-component (dx, dy, d_yaw).
# d_yaw has a much smaller range than dx, dy, so component-wise std is needed.

# State stats (for both current and future states — must match encoder input)
S_all  = X_states.reshape(-1, 6)   # same stats for future states
S_mean = S_all.mean(0).astype(np.float32)
S_std  = S_all.std(0).astype(np.float32) + 1e-8

# Trajectory stats
T_mean = Y_traj.mean(0).astype(np.float32)
T_std  = Y_traj.std(0).astype(np.float32) + 1e-8

print('State normalisation stats:')
for i, (mn, sd) in enumerate(zip(S_mean, S_std)):
    names = ['sin(yaw)', 'cos(yaw)', 'vx', 'vy', 'ax', 'ay']
    print(f'  {names[i]:10s}: mean={mn:+.3f}  std={sd:.3f}')

print(f'\nTraj mean range: [{T_mean.min():.4f}, {T_mean.max():.4f}]')
print(f'Traj std  range: [{T_std.min():.4f}, {T_std.max():.4f}]')

# Normalize
X_norm    = (X_states  - S_mean) / S_std              # (N, 6)
# Future states: each of 16 time steps uses the SAME per-feature stats
XF_norm   = (X_futures - S_mean) / S_std              # (N, 16, 6)
Y_norm    = (Y_traj    - T_mean) / T_std              # (N, 48)

# Train / val split (80/10/10 by log file already handled; here: by window)
np.random.seed(42)
n       = len(X_norm)
perm    = np.random.permutation(n)
n_tr    = int(0.8 * n)
n_va    = int(0.9 * n)
tr_idx  = perm[:n_tr]
va_idx  = perm[n_tr:n_va]

X_tr  = torch.tensor(X_norm[tr_idx],  dtype=torch.float32)
XF_tr = torch.tensor(XF_norm[tr_idx], dtype=torch.float32)
Y_tr  = torch.tensor(Y_norm[tr_idx],  dtype=torch.float32)

X_va  = torch.tensor(X_norm[va_idx],  dtype=torch.float32)
XF_va = torch.tensor(XF_norm[va_idx], dtype=torch.float32)
Y_va  = torch.tensor(Y_norm[va_idx],  dtype=torch.float32)

print(f'\nTrain: {len(X_tr):,}  |  Val: {len(X_va):,}')

# ── Action sequences — hoisted so Cell 6 (consistency check) can run
# independently of Cell 5 (training). If Cell 5 is skipped, Cell 6 still has
# Y_actions_va in scope without a NameError.
# Shape: (N, 16, 3) — each step's normalised (dx, dy, d_yaw).
Y_actions_tr = Y_tr.reshape(-1, FUTURE_STEPS, 3)   # (N_tr, 16, 3)
Y_actions_va = Y_va.reshape(-1, FUTURE_STEPS, 3)   # (N_va, 16, 3)
print(f'Action sequences: tr={tuple(Y_actions_tr.shape)}  va={tuple(Y_actions_va.shape)}')

In [ ]:
# Cell 4 — MILEPolicy architecture
#
# Three components, all trained jointly:
#
# 1. Encoder  E: R^6 → R^64
#    Maps normalised ego state to latent z.
#    Simple MLP (6→128→64). Deterministic (no VAE in this version).
#
# 2. World Model  T: R^64 × R^3 → R^64
#    GRU cell that takes (z_t, a_t) and outputs z_{t+1}.
#    a_t = first step of the expert trajectory = (dx_0, dy_0, d_yaw_0) [normalised].
#    WHY GRU (not MLP): GRU has memory — it accumulates gradient signal
#    from the 16-step consistency rollout without vanishing gradient problems.
#    WHY GRUCell (not GRU module): we roll out manually to access z at each step.
#
# 3. Policy  π: R^64 → R^48
#    Maps latent z_t to the full 16-step trajectory.
#    MLP (64→128→256→48).
#    WHY not use world model rollout at inference:
#    The world model is trained to predict the NEXT state, not the full 16-step
#    horizon. For inference, reading the trajectory from z_t directly is
#    equivalent to running the world model 16 times — but faster and doesn't
#    accumulate world-model prediction errors.
#
# Training objective:
#   z_t = E(state_t)
#   traj_pred = π(z_t)
#   L_imit = MSE(traj_pred, traj_gt)
#
#   for j = 0 ... 15:
#       a_j = traj_gt_normed[j, :3]   (use GT action for world model training)
#       z_{j+1}_pred = T(z_j, a_j)
#       z_{j+1}_true = E(state_{t+j+1})
#       L_cons += MSE(z_{j+1}_pred, z_{j+1}_true)
#
#   L_total = L_imit + BETA * (L_cons / 16)
#
# WHY condition world model on GT action (not predicted action) during training:
#   Teacher forcing. Using predicted actions during training creates a chicken-and-egg
#   dependency — the world model prediction quality depends on policy quality and vice
#   versa. Teacher forcing trains each component cleanly. At inference, the policy
#   only reads z_t, not the world model.
#
# REF: Hu et al. (2022) MILE Section 3.2 uses a similar teacher-forcing approach
#      with the full ELBO objective. We simplify to L2 for scalar state inputs.

class MILEPolicy(nn.Module):
    """
    MILE-inspired world model policy.

    Encoder:     6 → 128 → 64  (state → latent)
    World model: GRUCell(64 + 3, 64)  (latent + action → next latent)
    Policy:      64 → 128 → 256 → 48  (latent → trajectory)

    Parameters: ~73K  (BC MLP: ~260K — smaller because latent dim is only 64)
    """

    def __init__(
        self,
        state_dim:  int = 6,
        latent_dim: int = LATENT_DIM,
        act_dim:    int = 3,           # (dx, dy, d_yaw) per step
        out_dim:    int = FUTURE_STEPS * 3,
    ):
        super().__init__()

        # ── Encoder ────────────────────────────────────────────────────────
        self.encoder = nn.Sequential(
            nn.Linear(state_dim,  128), nn.ReLU(inplace=True),
            nn.Linear(128, latent_dim),
            nn.LayerNorm(latent_dim),   # WHY LayerNorm: stabilises latent space,
                                        # prevents the consistency loss from collapsing
                                        # all z to a single point.
        )

        # ── World model ────────────────────────────────────────────────────
        # GRUCell: input = [z_t || a_t] (concat), hidden = z_t
        self.world_model = nn.GRUCell(latent_dim + act_dim, latent_dim)

        # ── Policy ─────────────────────────────────────────────────────────
        self.policy = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(inplace=True),
            nn.Linear(128, 256),        nn.ReLU(inplace=True),
            nn.Linear(256, out_dim),
        )

    def encode(self, state: torch.Tensor) -> torch.Tensor:
        """state (B, 6) → latent (B, 64)"""
        return self.encoder(state)

    def step_world(self, z: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        """(z: B×64, action: B×3) → z_next: B×64"""
        inp = torch.cat([z, action], dim=-1)   # (B, 67)
        return self.world_model(inp, z)

    def predict_trajectory(self, z: torch.Tensor) -> torch.Tensor:
        """z (B, 64) → trajectory (B, 48)"""
        return self.policy(z)

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        """End-to-end: state (B, 6) → trajectory (B, 48). Used at inference."""
        return self.predict_trajectory(self.encode(state))


# ── Parameter count ───────────────────────────────────────────────────────────
model_mile = MILEPolicy().to(DEVICE)
n_params   = sum(p.numel() for p in model_mile.parameters())

from planners import BCPolicy
bc_params = sum(p.numel() for p in BCPolicy().parameters())
print(f'MILEPolicy parameters: {n_params:,}')
print(f'BCPolicy   parameters: {bc_params:,}')
print(f'MILE/BC ratio: {n_params/bc_params:.2f}x')

# Forward sanity check
z_test   = model_mile.encode(torch.zeros(4, 6).to(DEVICE))
traj_out = model_mile.predict_trajectory(z_test)
print(f'\nz shape:    {z_test.shape}   (expect (4, 64))')
print(f'traj shape: {traj_out.shape}  (expect (4, 48))')


In [ ]:
# Cell 5 — Training loop (imitation + consistency)
#
# This is the critical difference from BC/BEV training.
# After computing L_imit (standard MSE on the predicted trajectory),
# we run a 16-step world model rollout using TEACHER FORCING (GT actions)
# and penalise the difference between predicted and actual next latents.
#
# The consistency rollout builds a computational graph through 16 GRUCell steps.
# The gradient flows back through all 16 steps to update both the world model
# and the encoder simultaneously. This is why gradient clipping is essential —
# 16-step BPTT produces large gradients.
#
# NOTE: Y_actions_tr / Y_actions_va are defined in Cell 3 (normalization) so
# that Cell 6 (consistency check) can run independently of this training cell.

EPOCHS     = 30
BATCH_SIZE = 512
LR         = 1e-3

optimizer = torch.optim.Adam(model_mile.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=4, factor=0.5, verbose=True
)

best_val        = float('inf')
train_losses_imit   = []
train_losses_cons   = []
val_losses          = []

print(f'Training MILEPolicy for {EPOCHS} epochs ...')
print(f'L_total = L_imit + {BETA} * L_consistency')

for epoch in range(EPOCHS):
    model_mile.train()
    perm     = torch.randperm(len(X_tr))
    ep_imit  = 0.0
    ep_cons  = 0.0

    for start in range(0, len(X_tr), BATCH_SIZE):
        batch   = perm[start : start + BATCH_SIZE]
        s_b     = X_tr[batch].to(DEVICE)          # (B, 6)
        sf_b    = XF_tr[batch].to(DEVICE)         # (B, 16, 6)
        y_b     = Y_tr[batch].to(DEVICE)          # (B, 48)
        a_b     = Y_actions_tr[batch].to(DEVICE)  # (B, 16, 3)

        # ── Encode current state ────────────────────────────────────────
        z_t = model_mile.encode(s_b)              # (B, 64)

        # ── Imitation loss ──────────────────────────────────────────────
        traj_pred = model_mile.predict_trajectory(z_t)
        L_imit    = nn.functional.mse_loss(traj_pred, y_b)

        # ── Consistency loss (16-step teacher-forced world model rollout) ─
        # Teacher forcing: use GT action a_j to drive the world model.
        # WHY: gradient flows cleanly through world model without depending on
        #      policy output quality at this stage of training.
        L_cons = torch.tensor(0.0, device=DEVICE)
        z_j    = z_t
        for j in range(FUTURE_STEPS):
            z_j_pred = model_mile.step_world(z_j, a_b[:, j, :])   # predicted z_{j+1}
            z_j_true = model_mile.encode(sf_b[:, j, :])            # actual z_{j+1}

            # WHY detach z_j_true:
            # The encoder is being trained twice per step — once for the
            # imitation path and once here. Detaching z_j_true stops the
            # consistency loss from updating the encoder based on its own
            # output, which would create a trivial collapse (encoder always
            # predicts 0). The encoder is still updated via L_imit gradients.
            L_cons = L_cons + nn.functional.mse_loss(z_j_pred, z_j_true.detach())

            z_j = z_j_pred   # advance: next step uses predicted (not GT) latent
            # WHY use predicted z (not GT): we want the world model to learn
            # to chain correctly, not just do single-step predictions.

        L_cons   = L_cons / FUTURE_STEPS
        L_total  = L_imit + BETA * L_cons

        optimizer.zero_grad()
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model_mile.parameters(), max_norm=1.0)
        optimizer.step()

        B = len(batch)
        ep_imit += L_imit.item() * B
        ep_cons += L_cons.item() * B

    ep_imit /= len(X_tr)
    ep_cons /= len(X_tr)

    # ── Validation ────────────────────────────────────────────────────────
    model_mile.eval()
    val_imit = 0.0
    with torch.no_grad():
        for start in range(0, len(X_va), BATCH_SIZE):
            end   = min(start + BATCH_SIZE, len(X_va))
            s_v   = X_va[start:end].to(DEVICE)
            y_v   = Y_va[start:end].to(DEVICE)
            z_v   = model_mile.encode(s_v)
            pred_v = model_mile.predict_trajectory(z_v)
            val_imit += nn.functional.mse_loss(pred_v, y_v).item() * (end - start)
    val_imit /= len(X_va)

    scheduler.step(val_imit)
    train_losses_imit.append(ep_imit)
    train_losses_cons.append(ep_cons)
    val_losses.append(val_imit)

    if val_imit < best_val:
        best_val = val_imit
        torch.save({
            'model':  model_mile.state_dict(),
            'S_mean': S_mean, 'S_std': S_std,
            'T_mean': T_mean, 'T_std': T_std,
        }, CKPT_MILE)

    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:3d}:  '
              f'L_imit={ep_imit:.5f}  L_cons={ep_cons:.5f}  '
              f'val={val_imit:.5f}  best={best_val:.5f}')

print(f'\nMILEPolicy saved to {CKPT_MILE}')
print(f'Best val L_imit: {best_val:.5f}')
print()
print('Check: L_cons should decrease smoothly (world model learning dynamics).')
print('If L_cons stays high, the world model has not learned to predict transitions.')
print('Likely cause: BETA too high — try BETA=0.1 and retrain.')

In [ ]:
# Cell 5b — Loss curve diagnostic
#
# Three curves on one plot:
#   train_imit  : imitation loss (MSE traj_pred vs traj_gt)
#   train_cons  : consistency loss (MSE z_pred vs z_true, averaged over 16 steps)
#   val_imit    : validation imitation loss
#
# What to look for:
#   train_imit and val_imit should both decrease and converge.
#   train_cons should also decrease — world model is learning dynamics.
#   Gap (val > train) > 2x at convergence → overfitting. Fix: reduce EPOCHS or add dropout.
#   train_cons not decreasing → world model stuck. Fix: reduce BETA or increase LR.
#   train_imit < 0.01 but val > 0.05 → encoder memorised, not generalised. Try weight decay.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

epochs_x = list(range(1, len(train_losses_imit) + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: Imitation loss ─────────────────────────────────────────────────────
ax1.plot(epochs_x, train_losses_imit, label='train L_imit', color='steelblue')
ax1.plot(epochs_x, val_losses,        label='val   L_imit', color='darkorange', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE (normalised)'); ax1.set_title('Imitation Loss')
ax1.legend(); ax1.grid(alpha=0.3)

# ── Right: Consistency loss ──────────────────────────────────────────────────
ax2.plot(epochs_x, train_losses_cons, label='train L_cons', color='seagreen')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MSE (latent space)'); ax2.set_title('Consistency Loss (World Model)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
loss_curve_path = Path('checkpoints/mile_loss_curve.png')
loss_curve_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(loss_curve_path, dpi=120)
plt.close(fig)
print(f'Saved: {loss_curve_path}')

# ── Diagnosis ────────────────────────────────────────────────────────────────
final_tr  = train_losses_imit[-1]
final_va  = val_losses[-1]
final_con = train_losses_cons[-1]
ratio     = final_va / (final_tr + 1e-9)
print(f'\nFinal  train L_imit:  {final_tr:.5f}')
print(f'Final  val   L_imit:  {final_va:.5f}  (ratio val/train = {ratio:.2f}x)')
print(f'Final  train L_cons:  {final_con:.5f}')
print()
if ratio > 2.0:
    print('WARNING: val/train > 2x — overfitting. Consider fewer epochs or weight decay.')
elif ratio < 1.05:
    print('OK: train and val curves tight — no significant overfitting.')
else:
    print('OK: mild generalisation gap — normal for this dataset size.')
if final_con > 0.5:
    print('WARNING: L_cons still high — world model has not converged.')
    print('         Try: BETA=0.1, more epochs, or smaller LR.')
else:
    print('OK: L_cons converged — world model learned dynamics.')

In [ ]:
# Cell 6 — World model consistency sanity check
#
# Before running ADE/FDE eval, verify the world model actually learned dynamics.
# Test: start at state_t, use GT actions to roll out world model 16 steps,
# compare predicted z_{t+16} to actual encoded z_{t+16}.
#
# If the world model learned dynamics:
#   prediction error should be low at step 1 and grow slowly with horizon.
# If it hasn't:
#   error is large even at step 1 (world model is predicting noise).

ckpt_mile = torch.load(CKPT_MILE, map_location='cpu', weights_only=False)
model_eval = MILEPolicy().eval()
model_eval.load_state_dict(ckpt_mile['model'])

n_check  = min(500, len(X_va))
rng      = np.random.default_rng(42)
check_idx = rng.choice(len(X_va), n_check, replace=False)

step_errors = np.zeros(FUTURE_STEPS)   # mean L2 error per step

with torch.no_grad():
    for i in check_idx:
        s_0   = X_va[i].unsqueeze(0)           # (1, 6)
        sf_16 = XF_va[i].unsqueeze(0)          # (1, 16, 6)
        a_16  = Y_actions_va[i].unsqueeze(0)   # (1, 16, 3)

        z_j = model_eval.encode(s_0)
        for j in range(FUTURE_STEPS):
            z_j = model_eval.step_world(z_j, a_16[:, j, :])
            z_true = model_eval.encode(sf_16[:, j, :])
            step_errors[j] += (z_j - z_true).pow(2).mean().item()

step_errors /= n_check

print('World model prediction error (MSE in latent space) per horizon step:')
print(f'  Step  1 (0.1s): {step_errors[0]:.4f}')
print(f'  Step  4 (0.4s): {step_errors[3]:.4f}')
print(f'  Step  8 (0.8s): {step_errors[7]:.4f}')
print(f'  Step 16 (1.6s): {step_errors[15]:.4f}')
print()
print('Expected: errors grow monotonically with horizon (compounding prediction error).')
print('If step 1 error > 0.5, the world model has not converged. Try more epochs.')
print('If all steps ~ same error, the world model is outputting a constant (degenerate).')


In [ ]:
# Cell 7 — Open-loop ADE / FDE comparison: BC vs MILE
#
# Evaluates on the same 2,000 validation windows as bc_pipeline.ipynb.
# Both models predict from the SAME ground-truth ego state.
#
# Expected result:
#   MILE open-loop ADE should be CLOSE to BC (0.058m).
#   The policy branch of MILE is a simple MLP (64→128→256→48) which has LESS
#   capacity than BC MLP (6→256→256→256→48). So MILE may slightly underperform
#   BC in open-loop ADE. This is acceptable — the MILE advantage is in
#   closed-loop, where the world model enables better generalisation.
#
# If MILE ADE >> BC ADE (say, >0.1m): the imitation branch has insufficient
# capacity. Fix: increase LATENT_DIM or policy hidden size.

from planners import BCPolicy

ckpt_mile = torch.load(CKPT_MILE, map_location='cpu', weights_only=False)
mile_eval  = MILEPolicy().eval()
mile_eval.load_state_dict(ckpt_mile['model'])

ckpt_bc   = torch.load(CKPT_BC, map_location='cpu', weights_only=False)
bc_eval   = BCPolicy().eval()
bc_eval.load_state_dict(ckpt_bc['model'])

rng      = np.random.default_rng(42)
n_eval   = 2000
eval_idx = rng.choice(len(X_va), min(n_eval, len(X_va)), replace=False)

def eval_ade_fde(pred_np, gt_np):
    d = np.sqrt(np.sum((pred_np[:, :2] - gt_np[:, :2])**2, axis=1))
    return d.mean(), d[-1]

ade_mile, fde_mile = [], []
ade_bc,   fde_bc   = [], []

with torch.no_grad():
    for idx in eval_idx:
        gt_raw = (Y_va[idx].numpy() * T_std + T_mean).reshape(FUTURE_STEPS, 3)

        # MILE
        s_mile = X_va[idx].unsqueeze(0)
        pred_mile_norm = mile_eval(s_mile).squeeze(0).numpy()
        pred_mile = (pred_mile_norm * T_std + T_mean).reshape(FUTURE_STEPS, 3)

        # BC (uses its own normalisation stats)
        raw_state = X_states[va_idx[idx]]   # unnormalised state at this val window
        s_bc = torch.tensor(
            (raw_state - ckpt_bc['X_mean']) / ckpt_bc['X_std'], dtype=torch.float32
        ).unsqueeze(0)
        pred_bc_norm = bc_eval(s_bc).squeeze(0).numpy()
        pred_bc = (pred_bc_norm * ckpt_bc['Y_std'] + ckpt_bc['Y_mean']).reshape(FUTURE_STEPS, 3)

        a_m, f_m = eval_ade_fde(pred_mile, gt_raw)
        a_b, f_b = eval_ade_fde(pred_bc,   gt_raw)
        ade_mile.append(a_m); fde_mile.append(f_m)
        ade_bc.append(a_b);   fde_bc.append(f_b)

print(f"{'Policy':<12} {'ADE (m)':>10} {'FDE (m)':>10}")
print('-' * 34)
print(f"{'MILE':<12} {np.mean(ade_mile):>10.3f} {np.mean(fde_mile):>10.3f}")
print(f"{'BC MLP':<12} {np.mean(ade_bc):>10.3f} {np.mean(fde_bc):>10.3f}")
print()
print('Note: MILE open-loop ADE ≥ BC ADE is expected (smaller policy head).')
print('The world model advantage shows in CLOSED-LOOP, not open-loop.')


In [ ]:
# Cell 8 — MILEPlanner: AbstractPlanner wrapper
#
# Inference path: state_t → encoder → z_t → policy → trajectory
# The world model is NOT used at inference (see rationale in Cell 4).
#
# Architecture note: MILEPlanner is structurally identical to BCPlanner.
# The only differences are (1) it uses MILEPolicy instead of BCPolicy and
# (2) it applies state normalisation stats from the MILE checkpoint.
# The world model component is invisible at inference time.

import sys
sys.path.insert(0, '/Users/parvpatodia/nuplan-devkit')
sys.path.insert(0, '/Users/parvpatodia/Desktop/diffusion-policy-zoo/nuplan')

import numpy as np
import torch
from nuplan.common.actor_state.ego_state import EgoState
from nuplan.common.actor_state.state_representation import StateSE2, StateVector2D, TimePoint
from nuplan.planning.simulation.observation.observation_type import DetectionsTracks
from nuplan.planning.simulation.planner.abstract_planner import (
    AbstractPlanner, PlannerInitialization, PlannerInput,
)
from nuplan.planning.simulation.trajectory.interpolated_trajectory import InterpolatedTrajectory

DT_SIM = 0.1


class MILEPlanner(AbstractPlanner):
    """
    AbstractPlanner wrapper for MILEPolicy.
    Inference path: state → encoder → latent → policy → trajectory.
    World model is used during training only (not at inference).
    """

    def __init__(self, ckpt_path: str):
        self._ckpt_path = ckpt_path
        self._device    = torch.device('cpu')
        self._model     = None
        self._S_mean = self._S_std = None
        self._T_mean = self._T_std = None

    def name(self) -> str:
        return 'MILEPlanner'

    def observation_type(self):
        return DetectionsTracks

    def initialize(self, initialization: PlannerInitialization) -> None:
        ckpt = torch.load(self._ckpt_path, map_location=self._device, weights_only=False)
        self._model = MILEPolicy().to(self._device)
        self._model.load_state_dict(ckpt['model'])
        self._model.eval()
        self._S_mean = torch.tensor(ckpt['S_mean'], dtype=torch.float32)
        self._S_std  = torch.tensor(ckpt['S_std'],  dtype=torch.float32)
        self._T_mean = ckpt['T_mean']
        self._T_std  = ckpt['T_std']

    def _ego_features(self, ego: EgoState) -> np.ndarray:
        h   = ego.rear_axle.heading
        dcs = ego.dynamic_car_state
        return np.array([
            np.sin(h), np.cos(h),
            dcs.rear_axle_velocity_2d.x,
            dcs.rear_axle_velocity_2d.y,
            dcs.rear_axle_acceleration_2d.x,
            dcs.rear_axle_acceleration_2d.y,
        ], dtype=np.float32)

    def compute_planner_trajectory(self, current_input: PlannerInput) -> InterpolatedTrajectory:
        ego  = current_input.history.current_state[0]
        feat = self._ego_features(ego)

        # Normalise + forward pass
        x_t  = torch.tensor((feat - self._S_mean.numpy()) / self._S_std.numpy(),
                              dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            pred_norm = self._model(x_t).squeeze(0).numpy()
        pred = (pred_norm * self._T_std + self._T_mean).reshape(FUTURE_STEPS, 3)

        # Build trajectory (identical to BCPlanner)
        cx, cy    = ego.rear_axle.x, ego.rear_axle.y
        heading   = ego.rear_axle.heading
        cos_h, sin_h = np.cos(heading), np.sin(heading)
        dcs = ego.dynamic_car_state
        vx  = dcs.rear_axle_velocity_2d.x
        vy  = dcs.rear_axle_velocity_2d.y
        ax  = dcs.rear_axle_acceleration_2d.x
        ay  = dcs.rear_axle_acceleration_2d.y
        t0  = ego.time_point.time_us

        states = [ego]
        for j, (dx_e, dy_e, d_yaw) in enumerate(pred):
            wx    = cx + cos_h * dx_e - sin_h * dy_e
            wy    = cy + sin_h * dx_e + cos_h * dy_e
            w_yaw = heading + d_yaw
            states.append(EgoState.build_from_rear_axle(
                rear_axle_pose=StateSE2(wx, wy, w_yaw),
                rear_axle_velocity_2d=StateVector2D(vx, vy),
                rear_axle_acceleration_2d=StateVector2D(ax, ay),
                tire_steering_angle=0.0,
                time_point=TimePoint(t0 + int((j + 1) * DT_SIM * 1e6)),
                vehicle_parameters=ego.car_footprint.vehicle_parameters,
            ))
        return InterpolatedTrajectory(states)

print('MILEPlanner defined.')
print('Add MILEPlanner(str(CKPT_MILE)) to closed_loop_eval.py to run full eval.')


## Architecture analysis and extensions

### What the consistency loss enforces

After training, the encoder E and world model T satisfy (approximately):

    E(state_{t+1}) ≈ T(E(state_t), a_t)

This means the latent space is *structured* — moving through it with the world
model follows the actual ego state trajectory. BC has no such structure: its
latent space (the 256-dim hidden layer) is unstructured and may not generalise
to states not seen during training.

### Why MILE may help with covariate shift

In closed-loop, the ego visits novel states (drifted from the training distribution).
With BC, the policy is evaluated at an unstructured latent point that was never
visited during training → output is arbitrary.

With MILE, the world model provides a "path" from the last known good state to the
current novel state. If the encoder and world model have generalised, the latent at
the novel state will be *semantically close* to the latent at the last expert state,
even if the pixel/state values are different.

This is the core claim of the MILE paper (Section 4.3 in Hu et al. 2022). Our
simplified version tests this on scalar state rather than BEV images.

### Current limitations

1. **No BEV input**: this version uses only ego state. The MILE paper uses BEV images
   as the primary input. Adding BEV would require combining bev_cnn.ipynb rasterizer
   with this world model.

2. **Deterministic world model**: the full MILE uses a VAE for the world model
   (stochastic transitions). Our GRU is deterministic. For nuPlan scenarios that
   are nearly deterministic (single ego, no interactive agents), this is fine.

3. **No imagined rollout at inference**: we read trajectory from z_t directly.
   Full model-based planning would roll out the world model and select the
   trajectory that maximises an objective (CEM, MCTS).

### Closed-loop prediction

Expected result from closed-loop eval (Cell 7 in closed_loop_eval.py):
- MILEPlanner avg L2 should be lower than BCPlanner (49.4m)
- How much lower depends on how well the world model generalised
- A 20-40% improvement would be meaningful for a scalar-state model

To add MILEPlanner to the evaluation harness, update closed_loop_eval.py:
```python
from planners import MILEPlanner
run_simulation(MILEPlanner(str(CKPT_MILE)), SAVE_DIR, n_scenarios=3)
```

### Roadmap to full MILE
| Step | Description | Notebook |
|---|---|---|
| Done | Encoder + GRU world model + joint training | this notebook |
| Next | BEV input (replace 6-dim with 3×64×64) | combine with bev_cnn.ipynb |
| Phase 3 | Stochastic world model (VAE, ELBO) | mile_vae.ipynb |
| Phase 3 | Imagined rollout at inference (CEM) | mile_planning.ipynb |
